##### Copyright 2026 The MediaPipe Authors. All Rights Reserved.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a href="https://colab.research.google.com/github/google-ai-edge/mediapipe-samples/blob/main/examples/audio_transcriber/python/audio_transcriber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio Transcription with MediaPipe Tasks

In this notebook you will use the MediaPipe Tasks `AudioTranscriber` API to convert speech to text (automatic speech recognition, or ASR) entirely on device.

The sample uses [NVIDIA Parakeet TDT 0.6B v3](https://huggingface.co/litert-community/parakeet-tdt-0.6b-v3), a multilingual speech recognition model, converted to the LiteRT-LM format. You will:

1. Transcribe a whole audio clip in a single call (batch mode).
2. Transcribe the same clip by streaming it in small chunks, as you would with live microphone input.

## Preparation

Let's start with installing MediaPipe. The `AudioTranscriber` task requires `mediapipe` 1.2.0 or newer.

In [ ]:
%pip install -q "mediapipe>=1.2.0" numpy

Then download an off-the-shelf model. This example uses the stateful, int8-quantized Parakeet TDT 0.6B v3 model (~615 MB, [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/)), which processes audio in 5-second windows. The file is only downloaded if it isn't already present.

In [ ]:
import os
import urllib.request

model_url = 'https://huggingface.co/litert-community/parakeet-tdt-0.6b-v3/resolve/main/parakeet_tdt_0.6b_v3_5s_i8_stateful.litertlm'
model_path = model_url.split('/')[-1]

if not os.path.exists(model_path):
  print(f'Downloading {model_path}...')
  urllib.request.urlretrieve(model_url, model_path)
print(f'Model: {model_path} ({os.path.getsize(model_path) / 1e6:.0f} MB)')

Next, download a short `.wav` file of someone speaking (16 kHz, mono, 16-bit PCM) and play it back to check that it downloaded correctly.

In [ ]:
from IPython.display import Audio, display

audio_url = 'https://storage.googleapis.com/mediapipe-assets/speech_16000_hz_mono.wav'
audio_path = audio_url.split('/')[-1]

if not os.path.exists(audio_path):
  urllib.request.urlretrieve(audio_url, audio_path)
display(Audio(audio_path))

`AudioTranscriber` expects mono `float32` samples in the range `[-1.0, 1.0]`. The helper below uses Python's built-in `wave` module and NumPy to load any PCM `.wav` file: it converts integer samples to `float32` and averages stereo channels down to mono. It also returns the file's sample rate, which you will pass to the transcriber so that it can resample the audio to the 16 kHz the model expects.

In [ ]:
import wave

import numpy as np


def load_wav(path):
  """Loads a PCM .wav file as mono float32 samples in [-1.0, 1.0]."""
  with wave.open(path, 'rb') as wav_file:
    sample_rate = wav_file.getframerate()
    num_channels = wav_file.getnchannels()
    sample_width = wav_file.getsampwidth()
    frames = wav_file.readframes(wav_file.getnframes())

  if sample_width == 1:  # 8-bit PCM is unsigned.
    samples = (np.frombuffer(frames, dtype=np.uint8).astype(np.float32) - 128) / 128
  elif sample_width == 2:
    samples = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768
  elif sample_width == 4:
    samples = np.frombuffer(frames, dtype=np.int32).astype(np.float32) / 2**31
  else:
    raise ValueError(f'Unsupported sample width: {sample_width} bytes')

  # Average all channels to get mono audio.
  samples = samples.reshape(-1, num_channels).mean(axis=1)
  return samples, sample_rate


samples, sample_rate = load_wav(audio_path)
audio_duration = len(samples) / sample_rate
print(f'Loaded {len(samples)} samples at {sample_rate} Hz ({audio_duration:.2f} s)')

## Creating the AudioTranscriber

Configure the transcriber with `AudioTranscriberOptions`:

- `base_options`: the model path and the delegate (hardware) to run on. `Delegate.GPU` runs the model on the GPU where one is available; `Delegate.CPU` runs it on the CPU. If the GPU can't be initialized in your runtime, the code below falls back to the CPU.
- `sample_rate_hz`: the sample rate of the audio you will pass in.
- `enable_interim_results`: whether streaming transcription should also return interim (not yet final) hypotheses. This only affects streaming; batch transcription always returns final results.

Calling `prewarm()` initializes the inference engine up front, so that the first transcription isn't slowed down by one-time setup.

In [ ]:
from mediapipe.tasks.python.audio import audio_transcriber
from mediapipe.tasks.python.core import base_options

BaseOptions = base_options.BaseOptions
Delegate = BaseOptions.Delegate

delegate = 'GPU' #@param ['GPU', 'CPU']


def create_transcriber(delegate):
  options = audio_transcriber.AudioTranscriberOptions(
      base_options=BaseOptions(model_asset_path=model_path, delegate=delegate),
      sample_rate_hz=sample_rate,
      enable_interim_results=True,
  )
  return audio_transcriber.AudioTranscriber.create_from_options(options)


try:
  transcriber = create_transcriber(Delegate[delegate])
except RuntimeError as e:
  if delegate == 'CPU':
    raise
  print(f'GPU is not available ({e}), falling back to CPU.')
  delegate = 'CPU'
  transcriber = create_transcriber(Delegate.CPU)

transcriber.prewarm()
print(f'AudioTranscriber is ready (delegate: {delegate}).')

## Batch transcription

To transcribe a complete recording, pass all of its samples to `transcribe()` with `end_of_stream=True`. The result contains a list of `segments`, each with the recognized `transcript` and its start and end time in milliseconds. `result.text` joins all segments into a single transcript.

The real-time factor (RTF) is the processing time divided by the audio duration: values below 1.0 mean that transcription is faster than real time.

In [ ]:
import time

start = time.perf_counter()
result = transcriber.transcribe(samples, end_of_stream=True)
latency = time.perf_counter() - start

print(f'Transcript: {result.text}\n')
for segment in result.segments:
  print(f'[{segment.from_timestamp_ms / 1000:5.2f}s - {segment.to_timestamp_ms / 1000:5.2f}s] {segment.transcript}')
print(f'\nLatency: {latency * 1000:.0f} ms for {audio_duration:.2f} s of audio (RTF: {latency / audio_duration:.3f})')

## Streaming transcription

For live audio, such as a microphone feed, you transcribe audio as it arrives. Here you simulate this by slicing the clip into 1-second chunks and passing each chunk to `transcribe()` with `end_of_stream=False`. The transcriber keeps its state between calls, and returns:

- **interim** segments (`is_final=False`): the current best guess, which may still change as more audio arrives.
- **final** segments (`is_final=True`): text that won't change anymore.

The model needs some audio after a word before it commits to it, so a short clip may only produce interim results while streaming. When the stream ends, call `finish_stream()` to flush any buffered audio and get the remaining final segments. After that, the transcriber is ready for a new stream.

In [ ]:
chunk_size = sample_rate  # 1 second of audio per chunk.
final_segments = []

for chunk_start in range(0, len(samples), chunk_size):
  chunk = samples[chunk_start:chunk_start + chunk_size]
  partial_result = transcriber.transcribe(chunk, end_of_stream=False)
  stream_time = (chunk_start + len(chunk)) / sample_rate
  for segment in partial_result.segments:
    kind = 'final  ' if segment.is_final else 'interim'
    print(f'[{stream_time:4.1f}s audio] {kind}: {segment.transcript}')
  final_segments += [s for s in partial_result.segments if s.is_final]

final_result = transcriber.finish_stream()
for segment in final_result.segments:
  print(f'[end of stream] final  : {segment.transcript}')
final_segments += final_result.segments

print('\nFull transcript:', ' '.join(s.transcript for s in final_segments))

Finally, release the model when you are done.

In [ ]:
transcriber.close()